In [ ]:
import h5py
path = "./dataset/results_batch_00_sampling_0.hdf5"
with h5py.File(path, "r") as f:
    def show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"[DATASET] {name} shape={obj.shape} dtype={obj.dtype}")
        else:
            print(f"[GROUP  ] {name}")
    f.visititems(show)

import h5py
import numpy as np

KEYS = [
    "APs_complete","LANs_complete","angMomentum_complete","aratios_complete",
    "eccs_complete","energy_complete","incs_complete","masses_complete",
    "mratios_complete","periods_complete","positions_complete",
    "smaxes_complete","tAnos_complete","times_complete","velocities_complete"
]

def load_one_h5(path, T=None, dtype=np.float32):
    sl = slice(None) if T is None else slice(0, T)
    out = {}
    with h5py.File(path, "r") as f:
        for k in KEYS:
            if k == "times_complete":
                out[k] = f[k][:, sl].astype(dtype)              # (N,T)
            else:
                # 其它都是 (N,T,...) 或 (N,T,3,3)
                out[k] = f[k][:, sl, ...].astype(dtype)
    return out

paths = [
    "./dataset/results_batch_00_sampling_0.hdf5",
    "./dataset/results_batch_01_sampling_0.hdf5",
    "./dataset/results_batch_02_sampling_0.hdf5",
    "./dataset/results_batch_03_sampling_0.hdf5",
    "./dataset/results_batch_04_sampling_0.hdf5",
]

# optional: limit to first T steps for faster loading and prototyping
T = 100

dicts = [load_one_h5(p, T=T) for p in paths]

data = {k: np.concatenate([d[k] for d in dicts], axis=0) for k in KEYS}

print(data["positions_complete"].shape)  # (N_total, T, 3, 3)
print("data load complete")



[DATASET] APs_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] LANs_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] angMomentum_complete shape=(8000, 1001, 3) dtype=float64
[DATASET] aratios_complete shape=(8000, 1001, 1) dtype=float64
[DATASET] eccs_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] energy_complete shape=(8000, 1001, 1) dtype=float64
[DATASET] incs_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] masses_complete shape=(8000, 1001, 3) dtype=float64
[DATASET] mratios_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] periods_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] positions_complete shape=(8000, 1001, 3, 3) dtype=float64
[DATASET] smaxes_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] tAnos_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] times_complete shape=(8000, 1001) dtype=float64
[DATASET] velocities_complete shape=(8000, 1001, 3, 3) dtype=float64
(40000, 100, 3, 3)
data load complete


In [9]:
# check if we get the same data as before
N = 8000*5  # 5 files, each with 5000 systems
assert data["positions_complete"].shape == (N, T, 3, 3)
assert data["velocities_complete"].shape == (N, T, 3, 3)
assert data["times_complete"].shape == (N, T)


In [10]:
import pandas as pd
import torch

labels = pd.read_csv("dataset/completed_systems_sampling_0.txt", sep=r"\s+", header=None,
                 names=["index","num_batch","time_unbound","ignore_other_than-1","dignosis"])

# datatype
labels = labels.astype({"index":"int64","num_batch":"int64","time_unbound":"float32","ignore_other_than-1":"float32","dignosis":"int64"})

arr = labels.to_numpy()
print(labels.columns)
print(arr.shape, arr.dtype)
print(arr[:10, :5])  
#tensor = torch.from_numpy(arr)  
#print(tensor.shape, tensor.dtype)


Index(['index', 'num_batch', 'time_unbound', 'ignore_other_than-1',
       'dignosis'],
      dtype='str')
(40000, 5) float64
[[360.           0.          10.           1.2236681    0.        ]
 [179.           0.           5.           1.22473073   0.        ]
 [183.           0.           5.           1.24002266   0.        ]
 [361.           0.          20.           1.27451062   0.        ]
 [178.           0.          30.           1.24214554   0.        ]
 [349.           0.          15.           1.26477361   0.        ]
 [180.           0.          20.           1.25246263   0.        ]
 [176.           0.          20.           1.27163601   0.        ]
 [367.           0.          65.           1.30586576   0.        ]
 [182.           0.          30.           1.32230663   0.        ]]


In [11]:
def nan_report(data, T=200):
    rep = []
    N = next(iter(data.values())).shape[0]
    for k, a in data.items():
        b = a[:, :T, ...] if (a.ndim >= 2 and a.shape[1] >= T) else a
        bad = ~np.isfinite(b).all(axis=tuple(range(1, b.ndim)))
        rep.append((k, int(bad.sum()), N))
    return sorted(rep, key=lambda x: x[1], reverse=True)

for k, bad, N in nan_report(data, T)[:]:
    print(k, bad, "/", N)


periods_complete 16847 / 40000
tAnos_complete 14509 / 40000
positions_complete 254 / 40000
APs_complete 0 / 40000
LANs_complete 0 / 40000
angMomentum_complete 0 / 40000
aratios_complete 0 / 40000
eccs_complete 0 / 40000
energy_complete 0 / 40000
incs_complete 0 / 40000
masses_complete 0 / 40000
mratios_complete 0 / 40000
smaxes_complete 0 / 40000
times_complete 0 / 40000
velocities_complete 0 / 40000


In [12]:
import numpy as np

# 1) Sort `labels` so its row order matches the sample order in `data` (recommended)
labels = labels.sort_values(["num_batch", "index"], ascending=True).reset_index(drop=True)

# 2) Build a boolean mask of samples to KEEP (length = original N, e.g., 40000)
#    A sample is "bad" if it contains any NaN/Inf within the first T time steps.
bad_pos = ~np.isfinite(data["positions_complete"][:, :T, :, :]).all(axis=(1, 2, 3))
bad_vel = ~np.isfinite(data["velocities_complete"][:, :T, :, :]).all(axis=(1, 2, 3))
mask = ~(bad_pos | bad_vel)

# 3) Apply the same mask to both `data` and `labels` (dropped samples disappear from both)
data = {k: v[mask] for k, v in data.items()}
labels = labels.loc[mask].reset_index(drop=True)

# 4) Generate y AFTER cleaning (so y only corresponds to the kept samples)
#    Positive class: time_unbound is (approximately) 5000.0
y = np.isclose(labels["time_unbound"].to_numpy(), 5000.0, rtol=0, atol=1e-6).astype(np.int32)

# 5) Sanity check: y length should match the number of remaining samples in `data`
assert len(y) == data["times_complete"].shape[0]
print("After cleaning:", len(y), "pos:", int(y.sum()))

pos = int((y == 1).sum())
neg = int((y == 0).sum())
print("pos:", pos, "neg:", neg, "total:", len(y))




After cleaning: 39746 pos: 14203
pos: 14203 neg: 25543 total: 39746


In [13]:
import numpy as np

N = len(y)
test_size = 5000
rng = np.random.default_rng(42)

pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]

n_pos = int(round(test_size * len(pos_idx) / N))
n_neg = test_size - n_pos

n_pos = min(n_pos, len(pos_idx))
n_neg = min(n_neg, len(neg_idx))
remain = test_size - (n_pos + n_neg)
if remain > 0:
    if len(pos_idx) - n_pos >= remain:
        n_pos += remain
    else:
        n_neg += remain

test_pos = rng.choice(pos_idx, size=n_pos, replace=False)
test_neg = rng.choice(neg_idx, size=n_neg, replace=False)

test_idx = np.concatenate([test_pos, test_neg])
rng.shuffle(test_idx)

train_mask = np.ones(N, dtype=bool)
train_mask[test_idx] = False

data_train = {k: v[train_mask] for k, v in data.items()}
data_test  = {k: v[test_idx]    for k, v in data.items()}

labels_train = labels.loc[train_mask].reset_index(drop=True)
labels_test  = labels.loc[test_idx].reset_index(drop=True)

y_train = y[train_mask]
y_test  = y[test_idx]

print("train:", len(y_train), "pos:", int(y_train.sum()))
print("test :", len(y_test),  "pos:", int(y_test.sum()))


train: 34746 pos: 12416
test : 5000 pos: 1787


## SVM

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

class LinearSVM(nn.Module):
    """
    Linear SVM classifier (binary).
    Uses hinge loss: max(0, 1 - y * f(x)), with y in {-1, +1}.
    """
    def __init__(self, in_dim: int):
        super().__init__()
        self.linear = nn.Linear(in_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # returns raw score f(x) (shape: [B])
        return self.linear(x).squeeze(-1)

def hinge_loss(scores: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    scores: f(x), shape [B]
    y: labels in {-1, +1}, shape [B]
    """
    return torch.clamp(1.0 - y * scores, min=0.0).mean()

@torch.no_grad()
def predict(scores: torch.Tensor) -> torch.Tensor:
    # return predicted label in {0,1}
    return (scores >= 0).long()

def train_svm(
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    X_val: torch.Tensor | None = None,
    y_val: torch.Tensor | None = None,
    epochs: int = 100,
    batch_size: int = 1024,
    lr: float = 1e-2,
    weight_decay: float = 1e-4,  # L2 regularization strength (SVM style)
    device: str | torch.device = "cpu",
):
    """
    y_train can be {0,1} or {-1,+1}.
    """
    device = torch.device(device)
    X_train = X_train.to(device)
    y_train = y_train.to(device)

    # Convert labels to {-1, +1}
    if y_train.min() >= 0:
        y_train_svm = y_train * 2 - 1
    else:
        y_train_svm = y_train
    y_train_svm = y_train_svm.float()

    in_dim = X_train.shape[1]
    model = LinearSVM(in_dim).to(device)

    ds = TensorDataset(X_train, y_train_svm)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)

    def l2_penalty(m: nn.Module) -> torch.Tensor:
        w = m.linear.weight
        return (w * w).sum()

  
    for ep in range(1, epochs + 1):
        model.train()
        total = 0.0
        nb = 0

        for xb, yb in dl:
            scores = model(xb)
            loss = hinge_loss(scores, yb) + 0.5 * weight_decay * l2_penalty(model)

            opt.zero_grad(set_to_none=True)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # NEW

            opt.step()
            total += loss.item()
            nb += 1

        msg = f"epoch {ep:03d} | train_loss {(total/nb):.6f}"

        # optional validation
        if X_val is not None and y_val is not None:
            model.eval()
            Xv = X_val.to(device)
            yv = y_val.to(device)
            yv01 = yv.clone()
            if yv01.min() < 0:  # {-1,+1} -> {0,1}
                yv01 = (yv01 > 0).long()
            scores_v = model(Xv)
            pred_v = predict(scores_v)
            acc = (pred_v == yv01.long()).float().mean().item()
            msg += f" | val_acc {acc:.4f}"

        print(msg)

    return model


@torch.no_grad()
def accuracy(model, X, y, device="cpu"):
    model.eval()
    X = X.to(device)
    y = y.to(device)

    # y can be {0,1} or {-1,+1}; convert to {0,1} for acc
    if y.min() < 0:
        y01 = (y > 0).long()
    else:
        y01 = y.long()

    scores = model(X)                 # raw score f(x)
    pred = (scores >= 0).long()       # decision boundary at 0
    acc = (pred == y01).float().mean().item()
    return acc

In [17]:
X_train = build_features(data_train, T=T, add_time=True, use_geom=True)
X_test  = build_features(data_test,  T=T, add_time=True, use_geom=True)

print("X_train:", X_train.shape, "finite:", np.isfinite(X_train).all())
print("X_test :", X_test.shape,  "finite:", np.isfinite(X_test).all())

# flatten
X_train_2d = torch.from_numpy(X_train).float().reshape(X_train.shape[0], -1)
X_test_2d  = torch.from_numpy(X_test ).float().reshape(X_test.shape[0], -1)

y_train_t = torch.from_numpy(y_train).long()
y_test_t  = torch.from_numpy(y_test ).long()

# ===== 标准化：只用 train 计算 mean/std =====
eps = 1e-8
mu = X_train_2d.mean(dim=0, keepdim=True)
std = X_train_2d.std(dim=0, keepdim=True).clamp_min(eps)

X_train_2d = (X_train_2d - mu) / std
X_test_2d  = (X_test_2d  - mu) / std

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = train_svm(
    X_train=X_train_2d,
    y_train=y_train_t,
    X_val=None,
    y_val=None,
    device=device
)

train_acc = accuracy(model, X_train_2d, y_train_t, device=device)
test_acc  = accuracy(model, X_test_2d,  y_test_t,  device=device)

print("train_acc:", train_acc)
print("test_acc :", test_acc)


X_train: (34746, 36, 100) finite: True
X_test : (5000, 36, 100) finite: True
Using device: cuda
epoch 001 | train_loss 0.848222
epoch 002 | train_loss 0.778142
epoch 003 | train_loss 0.571249
epoch 004 | train_loss 0.525789
epoch 005 | train_loss 0.529648
epoch 006 | train_loss 0.521556
epoch 007 | train_loss 0.516043
epoch 008 | train_loss 0.534508
epoch 009 | train_loss 0.545958
epoch 010 | train_loss 0.527773
epoch 011 | train_loss 0.541351
epoch 012 | train_loss 0.558626
epoch 013 | train_loss 0.532895
epoch 014 | train_loss 0.545292
epoch 015 | train_loss 0.559332
epoch 016 | train_loss 0.810355
epoch 017 | train_loss 0.774918
epoch 018 | train_loss 0.659857
epoch 019 | train_loss 0.536821
epoch 020 | train_loss 0.510687
epoch 021 | train_loss 0.560390
epoch 022 | train_loss 0.755556
epoch 023 | train_loss 0.885803
epoch 024 | train_loss 0.626826
epoch 025 | train_loss 0.536533
epoch 026 | train_loss 0.505606
epoch 027 | train_loss 0.500227
epoch 028 | train_loss 0.496558
epoch 02

## 1D CNN

In [ ]:
"""
End-to-end: build (B, C, T) features from your dataset + train a 1D-CNN classifier
using the first T=100 steps.

Features included (per time step):
A) Precomputed orbital/physical series (with angle sin/cos):
  - aratios (1)
  - mratios (2)
  - masses (3)
  - eccs (2)
  - smaxes (2)
  - periods (2)
  - energy (1)
  - angMomentum (3)
  - APs (2) -> sin/cos (4)
  - LANs (2) -> sin/cos (4)
  - tAnos (2) -> sin/cos (4)
  - incs (2) -> sin/cos (4)

B) Geometry from positions/velocities (translation/rotation invariant):
  - pairwise distances: r12, r13, r23 (3)
  - pairwise relative speed magnitudes: |v12|, |v13|, |v23| (3)
  - pairwise radial speeds: rdot12, rdot13, rdot23 (3)

C) Optional: normalized time channel (1)

Total channels ~ 32 (A) + 9 (B) + 1 (time) = ~42 channels.

You provide:
  - data: dict of numpy arrays with the keys shown in your printout
  - y: numpy array (N,) labels 0/1
"""

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

# ---------------------------
# Feature engineering helpers
# ---------------------------

def sincos(x: np.ndarray) -> np.ndarray:
    """x: (..., k) angles in radians. Return (..., 2k) with [sin, cos]."""
    x = x.astype(np.float32)
    return np.concatenate([np.sin(x), np.cos(x)], axis=-1)

def pairwise_geom(pos: np.ndarray, vel: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    pos: (N, T, 3, 3)  -> bodies, xyz
    vel: (N, T, 3, 3)
    Return geom features per step: (N, T, 9)
      [r12,r13,r23, v12,v13,v23, rdot12,rdot13,rdot23]
    All are coordinate-frame invariant (translation cancels in differences; rotation cancels in norms/dots).
    """
    # differences: dij = xi - xj
    x1, x2, x3 = pos[:, :, 0, :], pos[:, :, 1, :], pos[:, :, 2, :]
    v1, v2, v3 = vel[:, :, 0, :], vel[:, :, 1, :], vel[:, :, 2, :]

    d12 = x1 - x2
    d13 = x1 - x3
    d23 = x2 - x3

    u12 = v1 - v2
    u13 = v1 - v3
    u23 = v2 - v3

    r12 = np.linalg.norm(d12, axis=-1)  # (N,T)
    r13 = np.linalg.norm(d13, axis=-1)
    r23 = np.linalg.norm(d23, axis=-1)

    v12 = np.linalg.norm(u12, axis=-1)
    v13 = np.linalg.norm(u13, axis=-1)
    v23 = np.linalg.norm(u23, axis=-1)

    # radial speeds: rdot = (d · u) / |d|
    rdot12 = (np.sum(d12 * u12, axis=-1)) / (r12 + eps)
    rdot13 = (np.sum(d13 * u13, axis=-1)) / (r13 + eps)
    rdot23 = (np.sum(d23 * u23, axis=-1)) / (r23 + eps)

    geom = np.stack([r12, r13, r23, v12, v13, v23, rdot12, rdot13, rdot23], axis=-1)
    return geom.astype(np.float32)  # (N,T,9)

def build_features(data: dict, T: int = 200, add_time: bool = True, use_geom: bool = True) -> np.ndarray:
    sl = slice(0, T)
    N = data["aratios_complete"].shape[0]

    # ---- (A) Precomputed series ----
    ar = data["aratios_complete"][:, sl, :].astype(np.float32)         # (N,T,1)
    mr = data["mratios_complete"][:, sl, :].astype(np.float32)         # (N,T,2)
    ms = data["masses_complete"][:, sl, :].astype(np.float32)          # (N,T,3)
    ec = data["eccs_complete"][:, sl, :].astype(np.float32)            # (N,T,2)
    sm = data["smaxes_complete"][:, sl, :].astype(np.float32)          # (N,T,2)
    en = data["energy_complete"][:, sl, :].astype(np.float32)          # (N,T,1)
    am = data["angMomentum_complete"][:, sl, :].astype(np.float32)     # (N,T,3)

    # Angles -> sin/cos  (not use tAnos)
    ap  = sincos(data["APs_complete"][:, sl, :])                       # (N,T,4)
    lan = sincos(data["LANs_complete"][:, sl, :])                      # (N,T,4)
    inc = sincos(data["incs_complete"][:, sl, :])                      # (N,T,4)

    # delete periods_complete and tAnos_complete
    feats = [ar, mr, ms, ec, sm, en, am, ap, lan, inc]

    # ---- (B) Geometry from raw positions/velocities (invariant) ----
    if use_geom:
        pos = data["positions_complete"][:, sl, :, :].astype(np.float32)   # (N,T,3,3)
        vel = data["velocities_complete"][:, sl, :, :].astype(np.float32)  # (N,T,3,3)
        geom = pairwise_geom(pos, vel)                                     # (N,T,9)
        feats.append(geom)

    # ---- (C) Time channel ----
    if add_time:
        t = np.linspace(0.0, 1.0, T, dtype=np.float32)[None, :, None]      # (1,T,1)
        t = np.repeat(t, N, axis=0)                                        # (N,T,1)
        feats.append(t)

    X = np.concatenate(feats, axis=-1)   # (N,T,C)
    X = X.transpose(0, 2, 1)             # (N,C,T)
    return X.astype(np.float32)
# ---------------------------
# Dataset / Model
# ---------------------------

class SeqDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.astype(np.float32), dtype=torch.float32).view(-1, 1)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

class CNN1D(nn.Module):
    """
    A small, strong baseline 1D CNN for (B, C, T).
    """
    def __init__(self, in_ch: int):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(in_ch, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=5, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),      
        )

    def forward(self, x):
        return self.head(self.backbone(x))

# ---------------------------
# Training / Evaluation
# ---------------------------

def standardize_channels_train_only(X_tr: np.ndarray, X_te: np.ndarray, eps: float = 1e-6):
    """
    X_*: (N,C,T)
    Standardize each channel using mean/std over (N,T) from training split only.
    """
    mu = X_tr.mean(axis=(0, 2), keepdims=True)
    sd = X_tr.std(axis=(0, 2), keepdims=True) + eps
    X_tr = (X_tr - mu) / sd
    X_te = (X_te - mu) / sd
    return X_tr, X_te, (mu, sd)

@torch.no_grad()
def eval_model(model, loader, device):
    model.eval()
    all_prob, all_y = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb).detach().cpu()
        prob = torch.sigmoid(logits).numpy().ravel()
        all_prob.append(prob)
        all_y.append(yb.numpy().ravel())
    all_prob = np.concatenate(all_prob)
    all_y = np.concatenate(all_y)

    pred = (all_prob >= 0.5).astype(np.int32)
    auc = roc_auc_score(all_y, all_prob)
    f1 = f1_score(all_y, pred)
    acc = accuracy_score(all_y, pred)
    return auc, f1, acc

def train_cnn_on_trainset(
    data_train: dict,
    y_train: np.ndarray,
    T: int = 100,
    epochs: int = 20,
    batch_size: int = 256,
    lr: float = 1e-3,
    add_time: bool = True,
    use_geom: bool = True,
):
    X_tr = build_features(data_train, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y_tr = np.asarray(y_train).astype(np.float32)
    assert X_tr.shape[0] == y_tr.shape[0]

    mu = X_tr.mean(axis=(0, 2), keepdims=True)
    sd = X_tr.std(axis=(0, 2), keepdims=True) + 1e-6
    X_tr = (X_tr - mu) / sd
    norm = (mu, sd)

    train_loader = DataLoader(SeqDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True, drop_last=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CNN1D(in_ch=X_tr.shape[1]).to(device)

    pos = float(y_tr.sum())
    neg = float(len(y_tr) - pos)
    pos_weight = torch.tensor([neg / (pos + 1e-8)], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_n = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

            bs = xb.size(0)
            total_loss += loss.item() * bs
            total_n += bs

        print(f"epoch {ep:02d} | train_loss {total_loss/total_n:.6f}")
    return model, norm




@torch.no_grad()
def test_cnn(
    model,
    data_test: dict,
    y_test: np.ndarray,
    norm,
    T: int = 100,
    batch_size: int = 512,
    add_time: bool = True,
    use_geom: bool = True,
):
    X_te = build_features(data_test, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y_te = np.asarray(y_test).astype(np.float32)

    mu, sd = norm
    X_te = (X_te - mu) / sd

    device = next(model.parameters()).device
    loader = DataLoader(SeqDataset(X_te, y_te), batch_size=batch_size, shuffle=False)

    auc, f1, acc = eval_model(model, loader, device)
    return auc, f1, acc


   

In [ ]:
model, norm = train_cnn_on_trainset(
    data_train=data_train,
    y_train=y_train,  
    T=T,
    epochs=50,
    batch_size=256,
    lr=1e-3,
    use_geom=True,
    add_time=True,
)

auc, f1, acc = test_cnn(
    model,
    data_test=data_test,
    y_test=y_test,
    norm=norm,
    T=T,
    use_geom=True,
    add_time=True,
)

print("TEST | AUC:", auc, "F1:", f1, "Acc:", acc)


epoch 01 | train_loss 0.507253
epoch 02 | train_loss 0.450893
epoch 03 | train_loss 0.431951
epoch 04 | train_loss 0.423730
epoch 05 | train_loss 0.410731
epoch 06 | train_loss 0.400092
epoch 07 | train_loss 0.391985
epoch 08 | train_loss 0.383100
epoch 09 | train_loss 0.381072
epoch 10 | train_loss 0.374164
epoch 11 | train_loss 0.367323
epoch 12 | train_loss 0.364068
epoch 13 | train_loss 0.354556
epoch 14 | train_loss 0.349006
epoch 15 | train_loss 0.341796
epoch 16 | train_loss 0.337424
epoch 17 | train_loss 0.332924
epoch 18 | train_loss 0.331724
epoch 19 | train_loss 0.328181
epoch 20 | train_loss 0.320383
epoch 21 | train_loss 0.312900
epoch 22 | train_loss 0.314955
epoch 23 | train_loss 0.310165
epoch 24 | train_loss 0.300925
epoch 25 | train_loss 0.298574
epoch 26 | train_loss 0.294256
epoch 27 | train_loss 0.286779
epoch 28 | train_loss 0.282058
epoch 29 | train_loss 0.279282
epoch 30 | train_loss 0.279176
epoch 31 | train_loss 0.273712
epoch 32 | train_loss 0.268823
epoch 33

## Attention

In [21]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding for (B, T, D)."""
    def __init__(self, d_model: int, max_len: int = 1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)  # (max_len, D)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)  # (max_len,1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)  # even
        pe[:, 1::2] = torch.cos(position * div_term)  # odd
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, D)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,T,D)
        T = x.size(1)
        return x + self.pe[:, :T, :]

class AttnClassifier(nn.Module):
    """
    Transformer encoder classifier for inputs (B, C, T).
    - token = one time step
    - output = sequence-level binary logit
    """
    def __init__(
        self,
        in_ch: int,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 3,
        dim_ff: int = 512,
        dropout: float = 0.1,
        pooling: str = "mean",  # "mean" or "cls"
        max_len: int = 1000,
    ):
        super().__init__()
        assert pooling in ("mean", "cls")

        self.pooling = pooling
        self.in_proj = nn.Linear(in_ch, d_model)

        if pooling == "cls":
            self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        else:
            self.cls = None

        self.pos = PositionalEncoding(d_model, max_len=max_len)
        self.drop = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,   # (B,T,D)
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(0.3),
            nn.Linear(d_model, 1),
        )

        # init
        nn.init.xavier_uniform_(self.in_proj.weight)
        nn.init.zeros_(self.in_proj.bias)
        if self.cls is not None:
            nn.init.normal_(self.cls, mean=0.0, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,C,T) -> (B,T,C)
        x = x.transpose(1, 2)
        x = self.in_proj(x)  # (B,T,D)

        if self.pooling == "cls":
            B = x.size(0)
            cls_tok = self.cls.expand(B, -1, -1)  # (B,1,D)
            x = torch.cat([cls_tok, x], dim=1)    # (B,T+1,D)

        x = self.pos(x)
        x = self.drop(x)
        x = self.encoder(x)  # (B,T(or T+1),D)

        if self.pooling == "cls":
            seq = x[:, 0, :]          # CLS
        else:
            seq = x.mean(dim=1)       # mean pooling over time

        return self.head(seq)         # (B,1)

def train_attn_on_trainset(
    data_train: dict,
    y_train: np.ndarray,
    T: int = 100,
    epochs: int = 50,
    batch_size: int = 64,
    lr: float = 1e-3,
    add_time: bool = True,
    use_geom: bool = True,
    d_model: int = 128,
    nhead: int = 4,
    num_layers: int = 2,
    dim_ff: int = 512,
    dropout: float = 0.1,
    pooling: str = "cls",
    use_amp: bool = True,
):
    """
    Train on (data_train, y_train) only.
    Returns: model, norm=(mu, sd) computed from train set only.
    """
    # Build features from TRAIN set
    X_tr = build_features(data_train, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y_tr = np.asarray(y_train).astype(np.float32)
    assert X_tr.shape[0] == y_tr.shape[0], "X_tr and y_tr must have same N"

    # Standardize using TRAIN only
    mu = X_tr.mean(axis=(0, 2), keepdims=True)
    sd = X_tr.std(axis=(0, 2), keepdims=True) + 1e-6
    X_tr = (X_tr - mu) / sd
    norm = (mu, sd)

    train_loader = DataLoader(
        SeqDataset(X_tr, y_tr),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AttnClassifier(
        in_ch=X_tr.shape[1],
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_ff=dim_ff,
        dropout=dropout,
        pooling=pooling,
        max_len=T + (1 if pooling == "cls" else 0) + 5,
    ).to(device)

    # pos_weight from TRAIN only
    pos = float(y_tr.sum())
    neg = float(len(y_tr) - pos)
    pos_weight = torch.tensor([neg / (pos + 1e-8)], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_n = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):
                logits = model(xb)
                loss = loss_fn(logits, yb)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            bs = xb.size(0)
            total_loss += float(loss.detach().cpu().item()) * bs
            total_n += bs

        print(f"epoch {ep:02d} | train_loss {total_loss/total_n:.6f}")

    return model, norm


@torch.no_grad()
def test_attn(
    model,
    data_test: dict,
    y_test: np.ndarray,
    norm,
    T: int = 100,
    batch_size: int = 256,
    add_time: bool = True,
    use_geom: bool = True,
):
    """
    Evaluate on external test set (data_test, y_test), using train norm (mu, sd).
    Returns: (auc, f1, acc)
    """
    X_te = build_features(data_test, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y_te = np.asarray(y_test).astype(np.float32)
    assert X_te.shape[0] == y_te.shape[0], "X_te and y_te must have same N"

    mu, sd = norm
    X_te = (X_te - mu) / sd

    device = next(model.parameters()).device
    test_loader = DataLoader(
        SeqDataset(X_te, y_te),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False
    )

    auc, f1, acc = eval_model(model, test_loader, device)
    return auc, f1, acc

In [22]:

model, norm = train_attn_on_trainset(
    data_train=data_train,
    y_train=y_train,
    T=T,
    epochs=50,
    batch_size=64,
    lr=1e-3,
    use_geom=True,
    add_time=True,
    pooling="cls",
    use_amp=True,
)


auc, f1, acc = test_attn(
    model,
    data_test=data_test,
    y_test=y_test,
    norm=norm,
    T=T,
    use_geom=True,
    add_time=True,
)

print(f"TEST | AUC {auc:.4f} | F1 {f1:.4f} | Acc {acc:.4f}")



/tmp/ipykernel_8645/1703861942.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
/tmp/ipykernel_8645/1703861942.py:155: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))
/tmp/ipykernel_8645/1703861942.py:166: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):


epoch 01 | train_loss 0.520554
epoch 02 | train_loss 0.453705
epoch 03 | train_loss 0.431369
epoch 04 | train_loss 0.422815
epoch 05 | train_loss 0.406071
epoch 06 | train_loss 0.394941
epoch 07 | train_loss 0.389551
epoch 08 | train_loss 0.380663
epoch 09 | train_loss 0.371831
epoch 10 | train_loss 0.364496
epoch 11 | train_loss 0.360635
epoch 12 | train_loss 0.353382
epoch 13 | train_loss 0.348830
epoch 14 | train_loss 0.342159
epoch 15 | train_loss 0.339987
epoch 16 | train_loss 0.334732
epoch 17 | train_loss 0.327280
epoch 18 | train_loss 0.329022
epoch 19 | train_loss 0.322185
epoch 20 | train_loss 0.316171
epoch 21 | train_loss 0.313143
epoch 22 | train_loss 0.311292
epoch 23 | train_loss 0.305475
epoch 24 | train_loss 0.301338
epoch 25 | train_loss 0.298453
epoch 26 | train_loss 0.294953
epoch 27 | train_loss 0.290407
epoch 28 | train_loss 0.286557
epoch 29 | train_loss 0.282753
epoch 30 | train_loss 0.282261
epoch 31 | train_loss 0.274718
epoch 32 | train_loss 0.271524
epoch 33